# Week 4: Colab bottle benchmark

This checks WideResNet-50 speed on the assigned Colab GPU. It runs only three training epochs, saves and reloads a checkpoint, and evaluates the bottle test images. It does not start the full study.

Choose **Runtime > Change runtime type > GPU**, then run the cells below in order. Use the `week4_bottle.zip` supplied with this notebook. It contains the project scripts and original bottle images, not the full dataset or personal files. No GitHub login is needed.

The bottle images come from MVTec AD and use [CC BY-NC-SA 4.0](https://www.mvtec.com/research-teaching/datasets/mvtec-ad). Keep this attribution with the package.

This short test uses Colab's temporary storage. Download the result at the end. Drive storage and automatic resume still need to be set up before a full training study.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## 1. Upload the package

Run this cell, click Choose Files, and select `week4_bottle.zip` from your computer. Wait for the upload and extraction to finish.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only week4_bottle.zip.')
name, payload = next(iter(uploaded.items()))
if hashlib.sha256(payload).hexdigest() != 'dd8b56ae63b2f9ae725bf4569c8f3e2ee647804a0553f176537c410531eaf71d':
    raise ValueError('This is not the package built for this notebook.')
project = Path('/content/week4-benchmark')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as archive:
    for member in archive.infolist():
        target = (project / member.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise ValueError('Invalid archive path.')
    archive.extractall(project)
manifest = json.loads((project / 'bundle_manifest.json').read_text())
for relative, expected in manifest.items():
    assert hashlib.sha256((project / relative).read_bytes()).hexdigest() == expected, relative
assert len(list((project / 'data/mvtec_ad/bottle/train/good').glob('*.png'))) == 209
assert len(list((project / 'data/mvtec_ad/bottle/test').glob('*/*.png'))) == 83
del uploaded, payload
print('Package checked: 209 training images and 83 test images.')


## 2. Install the environment

This may take several minutes and downloads several GB of packages. It installs the pinned project environment separately from Colab's notebook Python. Wait for the cell to finish before continuing. If it fails, stop here and share the error.


In [ ]:
import os, shutil, sys
if shutil.disk_usage('/content').free < 15 * 1024**3:
    raise RuntimeError('Need at least 15 GiB free for setup and the benchmark.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
os.environ['PATH'] = str(Path.home() / '.local/bin') + os.pathsep + os.environ['PATH']
subprocess.run([sys.executable, 'scripts/setup.py', '--device', 'cuda'], cwd=project, check=True)
python = str(project / '.venv/bin/python')
subprocess.run([python, '-c', 'import torch; print(torch.__version__); print(torch.cuda.get_device_name(0))'], check=True)


## 3. Run the short benchmark

Use a microbatch of four, matching the laptop test. The effective batch size stays at 32. The first epoch includes warm-up costs; the speed estimate uses the next two. Leave this cell running until it finishes. This is a hardware test, so the three-epoch AUROC is not a final model result.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os, subprocess

project = Path('/content/week4-benchmark')
python = str(project / '.venv/bin/python')
output = project / 'work' / ('colab-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
command = [python, '-u', 'scripts/week4_preflight.py',
           '--micro-batch-size', '4', '--epochs', '3', '--output', str(output)]
with subprocess.Popen(command, cwd=project, env=env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
if returncode:
    raise RuntimeError(f'Benchmark failed with exit code {returncode}. See the output above.')


## 4. Compare and download

The speed ratio compares the measured bottle training times. It is an estimate, not a guarantee for several days of training. The full-study estimate uses the training counts of all 15 categories even though only bottle is uploaded here. Send back the printed summary or the downloaded JSON file.


In [ ]:
result = json.loads((output / 'benchmark.json').read_text())
laptop = json.loads((project / 'reports/week4/preflight.json').read_text())
seconds = result['median_epoch_seconds_after_first']
ratio = laptop['median_epoch_seconds_after_first'] / seconds
print('Colab GPU:', result['gpu'])
print(f'Colab: {seconds:.2f} seconds per bottle epoch')
print(f"Laptop: {laptop['median_epoch_seconds_after_first']:.2f} seconds per bottle epoch")
print(f'Training speed ratio (laptop time / Colab time): {ratio:.2f}; above 1 means Colab is faster')
print(f"Estimated 45-run training time: {result['estimated_45_run_training_hours']:.1f} hours, plus setup/evaluation")
print(f"Peak allocated GPU memory: {result['gpu_peak_allocated_mib']:.0f} MiB")
print('Checkpoint reload matched:', result['checkpoint_reload_exact'])
print('No full study has been started.')
summary = Path('/content/colab_benchmark.json')
summary.write_text(json.dumps(result, indent=2))
files.download(str(summary))


When finished, disconnect the runtime using **Runtime > Disconnect and delete runtime**. Download your result first. This notebook has been checked locally, but the Colab installation and GPU run still need to be executed here.
